In [ ]:
import pandas as pd
data_delta = pd.DataFrame(['Wine', 'Depression', 'dmft-all', 'tourism-23457vs01', 
              'diu-ro10-cat', 'newthyroid1', 'page_blocks_1_3_vs_4'], columns=["DATA"])
data_delta["DELTA"] = [1, 0, 1, 1, 0, 3, 1]
data_delta

# 1. Compare methods [MAIN RESULTS] - with thre metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# (F1, ROC, PRAUC)
final_list = ['page_blocks_1_3_vs_4', 'broadwaymult3',
              'Depression', 'Wine', 'diu-ro10-cat','newthyroid1', 'dmft-all']

opt_delta = list(data_delta[data_delta["DATA"].isin(final_list)]["DELTA"])
print(len(final_list))
print(final_list)
print(opt_delta)
top = 1
j_list = [13, 16, 17]   # 9:Acc, 10:Pre, 11:Rec, 12:Spe, 13:F1, 14:GM, 15:BA, 16:ROCAUC, 17:PRAUC
metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']

In [ ]:
print("TOP {} Average".format(top))
for j in j_list:
    print("METRIC:", metrics[j-9])
    # org/smo/bsm/ada
    df = pd.read_csv('./Results/OSBA_result.csv')
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(1+5+5+5)  # org/smo/bsm/ada
    org_ind = int(base_ind+num_class*1)
    meth1_ind = int(base_ind+num_class*6)
    meth2_ind = int(base_ind+num_class*11)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'ORG_tr':[0 for i in range(top*len(final_list))],
                           'ORG_va':[0 for i in range(top*len(final_list))],
                           'ORG_t':[0 for i in range(top*len(final_list))],
                           'SMO_tr':[0 for i in range(top*len(final_list))],
                           'SMO_va':[0 for i in range(top*len(final_list))],
                           'SMO_t':[0 for i in range(top*len(final_list))],
                           'BSM_tr':[0 for i in range(top*len(final_list))],
                           'BSM_va':[0 for i in range(top*len(final_list))],
                           'BSM_t':[0 for i in range(top*len(final_list))],
                           'ADA_tr':[0 for i in range(top*len(final_list))],
                           'ADA_va':[0 for i in range(top*len(final_list))],
                           'ADA_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_org = df_i.iloc[:,base_ind:org_ind]         # i_th dataset original results (163 classifiers)
        df_i_meth1 = df_i.iloc[:,org_ind:meth1_ind]      # i_th dataset SMO results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset BSM results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:]             # i_th dataset ADA results (163 classifier X 5 Resam strategy)
        best_org = pd.DataFrame(df_i_org.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_org_best = df_i_org.loc[:,best_org] 
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1]
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        df_i_best = pd.concat([df_i_base,df_i_org_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,10] = list(df_i_best.iloc[[j-9],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,11] = list(df_i_best.iloc[[j],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,12] = list(df_i_best.iloc[[j+9],2+top+top+top+k])
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    df_average= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average[col] = [0 for i in range(len(final_list))]

    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == str(final_list[i])]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))
            
    # ROS/SEN/STM
    df = pd.read_csv('./Results/RSS_result.csv')
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)                
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(5+5+5)
    meth1_ind = int(base_ind+num_class*5)
    meth2_ind = int(base_ind+num_class*10)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'ROS_tr':[0 for i in range(top*len(final_list))],
                           'ROS_va':[0 for i in range(top*len(final_list))],
                           'ROS_t':[0 for i in range(top*len(final_list))],
                           'SEN_tr':[0 for i in range(top*len(final_list))],
                           'SEN_va':[0 for i in range(top*len(final_list))],
                           'SEN_t':[0 for i in range(top*len(final_list))],
                           'STM_tr':[0 for i in range(top*len(final_list))],
                           'STM_va':[0 for i in range(top*len(final_list))],
                           'STM_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_meth1 = df_i.iloc[:,base_ind:meth1_ind]     # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1] 
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        df_i_best = pd.concat([df_i_base,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])       
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    df_average_base= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average_base[col] = [0 for i in range(len(final_list))]

    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == str(final_list[i])]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average_base.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float)) 
            
    # X (deltas)
    df = pd.read_csv('./Results/X(deltas)_result.csv')
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(5+5+5+5)  # XEL with 4 deltas
    meth1_ind = int(base_ind+num_class*5)
    meth2_ind = int(base_ind+num_class*10)
    meth3_ind = int(base_ind+num_class*15)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'X(1.5)_tr':[0 for i in range(top*len(final_list))],
                           'X(1.5)_va':[0 for i in range(top*len(final_list))],
                           'X(1.5)_t':[0 for i in range(top*len(final_list))],
                           'X(2.0)_tr':[0 for i in range(top*len(final_list))],
                           'X(2.0)_va':[0 for i in range(top*len(final_list))],
                           'X(2.0)_t':[0 for i in range(top*len(final_list))],
                           'X(2.5)_tr':[0 for i in range(top*len(final_list))],
                           'X(2.5)_va':[0 for i in range(top*len(final_list))],
                           'X(2.5)_t':[0 for i in range(top*len(final_list))],
                           'X(3.0)_tr':[0 for i in range(top*len(final_list))],
                           'X(3.0)_va':[0 for i in range(top*len(final_list))],
                           'X(3.0)_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_meth1 = df_i.iloc[:,base_ind:meth1_ind]     # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:meth3_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth4 = df_i.iloc[:,meth3_ind:]             # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1] 
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        best_meth4 = pd.DataFrame(df_i_meth4.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth4_best = df_i_meth4.loc[:,best_meth4]
        df_i_best = pd.concat([df_i_base,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth4_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,10] = list(df_i_best.iloc[[j-9],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,11] = list(df_i_best.iloc[[j],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,12] = list(df_i_best.iloc[[j+9],2+top+top+top+k])
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    df_average_x= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average_x[col] = [0 for i in range(len(final_list))]
    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == str(final_list[i])]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average_x.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))
    optimal = []
    for i, row in df_average_x.iterrows():
        optimal.append(row[opt_delta[i]+1])
    df_average_x['X(meta)'] = optimal
    df_average_x
    
    # XE (deltas)
    df = pd.read_csv('./Results/XE(deltas)_result.csv')
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(5+5+5+5)  # XEL with 4 deltas
    meth1_ind = int(base_ind+num_class*5)
    meth2_ind = int(base_ind+num_class*10)
    meth3_ind = int(base_ind+num_class*15)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_tr':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_va':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_t':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_tr':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_va':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_t':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_tr':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_va':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_t':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_tr':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_va':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_meth1 = df_i.iloc[:,base_ind:meth1_ind]     # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:meth3_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth4 = df_i.iloc[:,meth3_ind:]             # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1] 
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        best_meth4 = pd.DataFrame(df_i_meth4.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth4_best = df_i_meth4.loc[:,best_meth4]
        df_i_best = pd.concat([df_i_base,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth4_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,10] = list(df_i_best.iloc[[j-9],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,11] = list(df_i_best.iloc[[j],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,12] = list(df_i_best.iloc[[j+9],2+top+top+top+k])
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    df_average_xe= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average_xe[col] = [0 for i in range(len(final_list))]
    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == str(final_list[i])]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average_xe.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))   
    optimal = []
    for i, row in df_average_xe.iterrows():
        optimal.append(row[opt_delta[i]+1])
    df_average_xe['XE(meta)'] = optimal
    df_average_xe

    # XEL (deltas)
    df = pd.read_csv('./Results/XEL(deltas)_result.csv')
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(5+5+5+5)  # XEL with 4 deltas
    meth1_ind = int(base_ind+num_class*5)
    meth2_ind = int(base_ind+num_class*10)
    meth3_ind = int(base_ind+num_class*15)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'XEL(1.5)_tr':[0 for i in range(top*len(final_list))],
                           'XEL(1.5)_va':[0 for i in range(top*len(final_list))],
                           'XEL(1.5)_t':[0 for i in range(top*len(final_list))],
                           'XEL(2.0)_tr':[0 for i in range(top*len(final_list))],
                           'XEL(2.0)_va':[0 for i in range(top*len(final_list))],
                           'XEL(2.0)_t':[0 for i in range(top*len(final_list))],
                           'XEL(2.5)_tr':[0 for i in range(top*len(final_list))],
                           'XEL(2.5)_va':[0 for i in range(top*len(final_list))],
                           'XEL(2.5)_t':[0 for i in range(top*len(final_list))],
                           'XEL(3.0)_tr':[0 for i in range(top*len(final_list))],
                           'XEL(3.0)_va':[0 for i in range(top*len(final_list))],
                           'XEL(3.0)_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_meth1 = df_i.iloc[:,base_ind:meth1_ind]     # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:meth3_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth4 = df_i.iloc[:,meth3_ind:]             # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1] 
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        best_meth4 = pd.DataFrame(df_i_meth4.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth4_best = df_i_meth4.loc[:,best_meth4]
        df_i_best = pd.concat([df_i_base,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth4_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,10] = list(df_i_best.iloc[[j-9],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,11] = list(df_i_best.iloc[[j],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,12] = list(df_i_best.iloc[[j+9],2+top+top+top+k])
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    df_average_xel= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average_xel[col] = [0 for i in range(len(final_list))]
    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == str(final_list[i])]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average_xel.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))   
    optimal = []
    for i, row in df_average_xel.iterrows():
        optimal.append(row[opt_delta[i]+1])
    df_average_xel['XEL(meta)'] = optimal
    df_average_xel
                  
    df_perform = pd.concat([df_average.iloc[:,:2], df_average_base.iloc[:,1:2]], axis=1)
    df_perform = pd.concat([df_perform, df_average.iloc[:,2:]], axis=1)
    df_perform = pd.concat([df_perform, df_average_base.iloc[:,2:]], axis=1)
#     df_perform = pd.concat([df_perform, df_average_x.iloc[:,-1]], axis=1)
#     df_perform = pd.concat([df_perform, df_average_xe.iloc[:,-1]], axis=1)
    df_perform = pd.concat([df_perform, df_average_xel.iloc[:,-1]], axis=1)
    print(df_perform.iloc[:,1:])
    methods = df_perform.columns[1:]
    print(pd.DataFrame(df_perform[methods].mean()).T)

    # visualization
    plt.figure(figsize=(12, 8))
    for i, row in df_perform.iterrows():
        plt.plot(methods, row[1:], marker='o', label=row['DATA'])
    avg_scores = df_perform[methods].mean()
    plt.plot(methods, avg_scores, marker='s', color='black', linewidth=3, linestyle='--', label='AVERAGE')
    
    plt.title('Performance({}) with top {} average'.format(metrics[j-9], top), fontsize=16)
    plt.xlabel('Methods', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.legend(title='Datasets', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 2. Ablation Study (with the optimal delta_max)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 다시한번
final_list = ['page_blocks_1_3_vs_4', 'broadwaymult3',
              'Depression', 'Wine', 'diu-ro10-cat','newthyroid1', 'dmft-all']
top = 1
j_list = [13, 16, 17]   # 9:Acc, 10:Pre, 11:Rec, 12:Spe, 13:F1, 14:GM, 15:BA, 16:ROCAUC, 17:PRAUC
metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']

In [ ]:
print("TOP {} Average".format(top))
for j in j_list:
    print("METRIC:", metrics[j-9])
    # X (deltas)
    df = pd.read_csv('./Results/X(deltas)_result.csv')
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(5+5+5+5)  # XEL with 4 deltas
    meth1_ind = int(base_ind+num_class*5)
    meth2_ind = int(base_ind+num_class*10)
    meth3_ind = int(base_ind+num_class*15)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'X(1.5)_tr':[0 for i in range(top*len(final_list))],
                           'X(1.5)_va':[0 for i in range(top*len(final_list))],
                           'X(1.5)_t':[0 for i in range(top*len(final_list))],
                           'X(2.0)_tr':[0 for i in range(top*len(final_list))],
                           'X(2.0)_va':[0 for i in range(top*len(final_list))],
                           'X(2.0)_t':[0 for i in range(top*len(final_list))],
                           'X(2.5)_tr':[0 for i in range(top*len(final_list))],
                           'X(2.5)_va':[0 for i in range(top*len(final_list))],
                           'X(2.5)_t':[0 for i in range(top*len(final_list))],
                           'X(3.0)_tr':[0 for i in range(top*len(final_list))],
                           'X(3.0)_va':[0 for i in range(top*len(final_list))],
                           'X(3.0)_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_meth1 = df_i.iloc[:,base_ind:meth1_ind]     # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:meth3_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth4 = df_i.iloc[:,meth3_ind:]             # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1] 
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        best_meth4 = pd.DataFrame(df_i_meth4.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth4_best = df_i_meth4.loc[:,best_meth4]
        df_i_best = pd.concat([df_i_base,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth4_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,10] = list(df_i_best.iloc[[j-9],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,11] = list(df_i_best.iloc[[j],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,12] = list(df_i_best.iloc[[j+9],2+top+top+top+k])
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    df_average_x= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average_x[col] = [0 for i in range(len(final_list))]
    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == str(final_list[i])]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average_x.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))
    optimal = []
    for i, row in df_average_x.iterrows():
        optimal.append(row[opt_delta[i]+1])
    df_average_x['X(meta)'] = optimal
    df_average_x
    
    # XE (deltas)
    df = pd.read_csv('./Results/XE(deltas)_result.csv')
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(5+5+5+5)  # XEL with 4 deltas
    meth1_ind = int(base_ind+num_class*5)
    meth2_ind = int(base_ind+num_class*10)
    meth3_ind = int(base_ind+num_class*15)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_tr':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_va':[0 for i in range(top*len(final_list))],
                           'XE(1.5)_t':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_tr':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_va':[0 for i in range(top*len(final_list))],
                           'XE(2.0)_t':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_tr':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_va':[0 for i in range(top*len(final_list))],
                           'XE(2.5)_t':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_tr':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_va':[0 for i in range(top*len(final_list))],
                           'XE(3.0)_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_meth1 = df_i.iloc[:,base_ind:meth1_ind]     # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:meth3_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth4 = df_i.iloc[:,meth3_ind:]             # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1] 
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        best_meth4 = pd.DataFrame(df_i_meth4.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth4_best = df_i_meth4.loc[:,best_meth4]
        df_i_best = pd.concat([df_i_base,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth4_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,10] = list(df_i_best.iloc[[j-9],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,11] = list(df_i_best.iloc[[j],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,12] = list(df_i_best.iloc[[j+9],2+top+top+top+k])
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    df_average_xe= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average_xe[col] = [0 for i in range(len(final_list))]
    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == str(final_list[i])]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average_xe.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))   
    optimal = []
    for i, row in df_average_xe.iterrows():
        optimal.append(row[opt_delta[i]+1])
    df_average_xe['XE(meta)'] = optimal
    df_average_xe

    # XEL (deltas)
    df = pd.read_csv('./Results/XEL(deltas)_result.csv')
    df.rename(columns={'Unnamed: 0':'Metrics'}, inplace=True)
    base_ind = 2
    num_class = (df.shape[1]-base_ind)/(5+5+5+5)  # XEL with 4 deltas
    meth1_ind = int(base_ind+num_class*5)
    meth2_ind = int(base_ind+num_class*10)
    meth3_ind = int(base_ind+num_class*15)
    metrics = ['Acc', 'Pre', 'Rec', 'Spe', 'F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']  
    df_res = pd.DataFrame({'DATA':[0 for i in range(top*len(final_list))],
                           'XEL(1.5)_tr':[0 for i in range(top*len(final_list))],
                           'XEL(1.5)_va':[0 for i in range(top*len(final_list))],
                           'XEL(1.5)_t':[0 for i in range(top*len(final_list))],
                           'XEL(2.0)_tr':[0 for i in range(top*len(final_list))],
                           'XEL(2.0)_va':[0 for i in range(top*len(final_list))],
                           'XEL(2.0)_t':[0 for i in range(top*len(final_list))],
                           'XEL(2.5)_tr':[0 for i in range(top*len(final_list))],
                           'XEL(2.5)_va':[0 for i in range(top*len(final_list))],
                           'XEL(2.5)_t':[0 for i in range(top*len(final_list))],
                           'XEL(3.0)_tr':[0 for i in range(top*len(final_list))],
                           'XEL(3.0)_va':[0 for i in range(top*len(final_list))],
                           'XEL(3.0)_t':[0 for i in range(top*len(final_list))]})

    for i in range(len(final_list)):        # in one dataset
        df_i = df[df["Dataset"] == str(final_list[i])]   # i_th dataset
        df_i_base = df_i.iloc[:,:base_ind]               # i_th dataset base
        df_i_meth1 = df_i.iloc[:,base_ind:meth1_ind]     # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth2 = df_i.iloc[:,meth1_ind:meth2_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth3 = df_i.iloc[:,meth2_ind:meth3_ind]    # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        df_i_meth4 = df_i.iloc[:,meth3_ind:]             # i_th dataset XEL results (163 classifier X 5 Resam strategy)
        best_meth1 = pd.DataFrame(df_i_meth1.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth1_best = df_i_meth1.loc[:,best_meth1] 
        best_meth2 = pd.DataFrame(df_i_meth2.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth2_best = df_i_meth2.loc[:,best_meth2]
        best_meth3 = pd.DataFrame(df_i_meth3.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth3_best = df_i_meth3.loc[:,best_meth3]
        best_meth4 = pd.DataFrame(df_i_meth4.iloc[j,:]).iloc[:,0].sort_values(ascending=False)[:top].index
        df_i_meth4_best = df_i_meth4.loc[:,best_meth4]
        df_i_best = pd.concat([df_i_base,df_i_meth1_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth2_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth3_best],axis=1)
        df_i_best = pd.concat([df_i_best,df_i_meth4_best],axis=1)
        for k in range(top):
            df_res.iloc[i*top+k:i*top+k+1,0] = final_list[i]
            df_res.iloc[i*top+k:i*top+k+1,1] = list(df_i_best.iloc[[j-9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,2] = list(df_i_best.iloc[[j],2+k])
            df_res.iloc[i*top+k:i*top+k+1,3] = list(df_i_best.iloc[[j+9],2+k])
            df_res.iloc[i*top+k:i*top+k+1,4] = list(df_i_best.iloc[[j-9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,5] = list(df_i_best.iloc[[j],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,6] = list(df_i_best.iloc[[j+9],2+top+k])
            df_res.iloc[i*top+k:i*top+k+1,7] = list(df_i_best.iloc[[j-9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,8] = list(df_i_best.iloc[[j],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,9] = list(df_i_best.iloc[[j+9],2+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,10] = list(df_i_best.iloc[[j-9],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,11] = list(df_i_best.iloc[[j],2+top+top+top+k])
            df_res.iloc[i*top+k:i*top+k+1,12] = list(df_i_best.iloc[[j+9],2+top+top+top+k])
    test_result = ['DATA']
    for col in df_res.columns:
        if col[-2:] == '_t':
            test_result.append(col)
    df_test = df_res[test_result]
    df_average_xel= pd.DataFrame({'DATA':[final_list[i] for i in range(len(final_list))]})
    for col in df_test.columns[1:]:
        df_average_xel[col] = [0 for i in range(len(final_list))]
    for i in range(len(final_list)):        # in one dataset
        df_i = df_test[df_test["DATA"] == str(final_list[i])]   # i_th dataset
        for col in df_test.columns[1:]:
            df_average_xel.loc[i,col] = np.average(df_i.loc[top*i:top*(i+1),col].astype(float))   
    optimal = []
    for i, row in df_average_xel.iterrows():
        optimal.append(row[opt_delta[i]+1])
    df_average_xel['XEL(meta)'] = optimal
    df_average_xel
     
    df_perform = pd.concat([df_average_x.iloc[:,0], df_average_x.iloc[:,-1]], axis=1)    
    df_perform = pd.concat([df_perform, df_average_xe.iloc[:,-1]], axis=1)
    df_perform = pd.concat([df_perform, df_average_xel.iloc[:,-1]], axis=1)
    print(df_perform.iloc[:,1:])
    methods = df_perform.columns[1:]
    print(pd.DataFrame(df_perform[methods].mean()).T)

    # visualization
    plt.figure(figsize=(12, 8))
    for i, row in df_perform.iterrows():
        plt.plot(methods, row[1:], marker='o', label=row['DATA'])
    avg_scores = df_perform[methods].mean()
    plt.plot(methods, avg_scores, marker='s', color='black', linewidth=3, linestyle='--', label='AVERAGE')
    
    plt.title('Performance({}) with top {} average'.format(metrics[j-9], top), fontsize=16)
    plt.xlabel('Methods', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.legend(title='Datasets', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 3. Yield_Rate Analysis

In [ ]:
deltas = ['1.5', '2', '2.5', '3']

In [ ]:
Final_Yield = pd.DataFrame(deltas, columns=['Delta'])
for data in final_list:        # in one dataset
    df_analysis_data = pd.DataFrame(deltas, columns=['Delta'])
    times = []
    yieldrates_EXT = []
    yieldrates_ENN = []
    yieldrates_LLM = []
    REPs = []
    for delta in deltas:       # in one delta
        df_runtime = pd.read_csv('./Generated Data/{}/XEL_Trinity_Large/(delta={}){}_XEL_time.csv'.format(data, delta, data))
        df_yieldrate_EXT = pd.read_csv('./Generated Data/{}/X/(delta={}){}_X_yield.csv'.format(data, delta, data))
        df_yieldrate_ENN = pd.read_csv('./Generated Data/{}/XE/(delta={}){}_XE_yield.csv'.format(data, delta, data))
        df_yieldrate_LLM = pd.read_csv('./Generated Data/{}/XEL_Trinity_Large/(delta={}){}_XEL_yield.csv'.format(data, delta, data))
        df_keptnumber = pd.read_csv('./Generated Data/{}/XEL_Trinity_Large/(delta={}){}_XEL_kept.csv'.format(data, delta, data))
        df_rep = df_runtime/df_keptnumber
        times.append(np.sum(np.sum(df_runtime)))
        yieldrates_EXT.append(np.average(df_yieldrate_EXT))
        yieldrates_ENN.append(np.average(df_yieldrate_ENN))
        yieldrates_LLM.append(np.average(df_yieldrate_LLM))
        REPs.append(np.average(df_rep))
    df_analysis_data['TIME'] = times
    df_analysis_data['YIELD_EXT'] = yieldrates_EXT
    df_analysis_data['YIELD_ENN'] = yieldrates_ENN
    df_analysis_data['YIELD_LLM'] = yieldrates_LLM
    df_analysis_data['REP'] = REPs
    print(data,'\n',df_analysis_data)
    
    # visualization
    methods = df_analysis_data.columns[2:5]
    plt.figure(figsize=(12, 8))   
    for i, row in df_analysis_data.iterrows():
        plt.plot(methods, row[2:5], marker='o', label=row['Delta'])
        
# #     plt.plot(methods, df_analysis_data['TIME'], marker='o', label='TIME')
#     plt.plot(methods, df_analysis_data['YIELD_EXT'], marker='o', label='YIELD_EXT')
#     plt.plot(methods, df_analysis_data['YIELD_ENN'], marker='o', label='YIELD_ENN')
#     plt.plot(methods, df_analysis_data['YIELD_LLM'], marker='o', label='YIELD_LLM')
# #     plt.plot(methods, df_analysis_data['REP'], marker='o', label='REP')

    plt.title('YIELD_RATE({})'.format(data), fontsize=16)
    plt.xlabel('Delta_max', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.ylim(0, 1.1)
    plt.legend(title='Delta', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Contruct the final dataframe
    Final_Yield = pd.concat([Final_Yield, df_analysis_data['YIELD_LLM']], axis=1)
    Final_Yield.rename(columns={'YIELD_LLM':f'{data}'}, inplace=True)
print(Final_Yield)

# visualization
methods = Final_Yield['Delta']
plt.figure(figsize=(12, 8))   
for i in range(len(final_list)):
    plt.plot(methods, Final_Yield.iloc[:,i+1], marker='o', label=Final_Yield.columns[i+1])
avg_scores = Final_Yield.iloc[:,1:].T.mean()
plt.plot(deltas, avg_scores, marker='s', color='black', linewidth=3, linestyle='--', label='AVERAGE')

plt.title('YIELD_RATE(All data)', fontsize=16)
plt.xlabel('Delta_max', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.ylim(0, 0.5)
plt.legend(title='Delta', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()